# Vector Database Generation

Objectives: 
- Chunk the documents into smaller texts
- Generate vector embeddings for each chunk
- Store the embeddings in a vector database

Remarks: To improve efficiency, the vector database is generated offline. This allows the application to simply load the precomputed embeddings, significantly reducing processing time.

In [ ]:
import os
import sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import chromadb

from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore

from src.module.chunking import Chunker
from src.module.constants import VECTOR_DB_DIR, VECTOR_DB_COLLECTION_NAME, PARSED_DOC_DIR

/Users/kckhoo/Documents/Personal/Take Home Assessments/ENest/.venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


## Step 1: Chunk documents

Two steps:
- (a) Split the Markdown documents into sections based on markdown headers
- (b) Split section-level texts into smaller text chunks with a fixed-size window

In [ ]:
# Initialize markdown parser
markdown_parser = MarkdownNodeParser(
    include_metadata=True,
    include_prev_next_rel=False
)

# Initialize sentence splitter
sentence_splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=128,
)

# Initialize chunking pipeline
chunker = Chunker(markdown_parser, sentence_splitter)

# Chunk documents using markdown parser and sentence splitter
text_nodes = chunker.chunk_docs_from_dir(PARSED_DOC_DIR)

## Step 2: Generate Vector Embeddings and Store in the Vector Database

In [ ]:
db = chromadb.PersistentClient(path=os.path.join(parent_dir, VECTOR_DB_DIR))
chroma_collection = db.get_or_create_collection(VECTOR_DB_COLLECTION_NAME)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(text_nodes, storage_context=storage_context)